# Multi-Agent Horse Racing Training

All horses share one policy and compete simultaneously. Checkpoints save to Google Drive so training auto-resumes after a runtime disconnect.

**How to use:**
1. Run all cells top to bottom
2. If runtime disconnects, just reconnect and run all cells again — it picks up from the latest checkpoint automatically

## Configuration

In [ ]:
# ============================================================
# CONFIGURATION — edit these before running
# ============================================================

# Training params
TOTAL_TIMESTEPS = 5_000_000    # total env steps to train
MIN_HORSES = 4                 # min horses per episode
MAX_HORSES = 12                # max horses per episode
MAX_STEPS = 5000               # max steps per episode

# Checkpoint dir on Google Drive (persists across disconnects)
CHECKPOINT_DIR = "/content/drive/MyDrive/hr-checkpoints/multi_agent"

# Resume: 'auto' finds the latest checkpoint, None starts fresh
RESUME_CHECKPOINT = 'auto'

# Logging
LOG_EVERY = 5                  # print stats every N iterations
SAVE_EVERY = 25                # save checkpoint every N iterations

## Setup

In [ ]:
# Mount Google Drive for persistent checkpoints
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone or update repo
import os

REPO_DIR = "/content/hr-simulation"
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/ue-too/hr-simulation.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q "ray[rllib]>=2.9.0" "gymnasium>=1.0.0" "pettingzoo>=1.24.0" "torch>=2.0.0" "onnx>=1.21.0" "onnxruntime>=1.24.4" "onnxscript>=0.6.2" psutil

# Install the project in editable mode
!pip install -q -e .

## Checkpoint Recovery

In [ ]:
import glob
from pathlib import Path

def find_latest_checkpoint(checkpoint_dir: str) -> str | None:
    """Find the most recent RLlib checkpoint in a directory tree.

    Scans for algorithm_state.pkl (RLlib new API) or .is_checkpoint (old API)
    and returns the parent directory path of the newest one.
    """
    markers = []
    for pattern in ["**/algorithm_state.pkl", "**/.is_checkpoint"]:
        markers.extend(glob.glob(os.path.join(checkpoint_dir, pattern), recursive=True))
    if not markers:
        return None
    # Sort by modification time, newest first
    markers.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    return str(Path(markers[0]).parent)

# Find checkpoint to resume from
checkpoint_path = None
if RESUME_CHECKPOINT == 'auto':
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    checkpoint_path = find_latest_checkpoint(CHECKPOINT_DIR)
    if checkpoint_path:
        print(f"Found checkpoint: {checkpoint_path}")
    else:
        print("No existing checkpoint found — starting fresh")
elif RESUME_CHECKPOINT:
    checkpoint_path = RESUME_CHECKPOINT
    print(f"Resuming from: {checkpoint_path}")
else:
    print("Starting fresh (RESUME_CHECKPOINT=None)")

## Training

In [ ]:
import time
import json
import torch
import psutil
import ray
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.policy.policy import PolicySpec

from horse_racing.rllib_env import HorseRacingRLlibEnv

# Detect hardware
num_gpus = 1 if torch.cuda.is_available() else 0
total_cpus = psutil.cpu_count()
num_workers = max(1, min(total_cpus - 1, 6))

# Find all tracks
track_paths = sorted(glob.glob("tracks/*.json"))
print(f"Tracks: {len(track_paths)}")
print(f"Hardware: {num_workers} CPU workers, {num_gpus} GPU")
print(f"Horses: {MIN_HORSES}-{MAX_HORSES}")
print(f"Target: {TOTAL_TIMESTEPS:,} timesteps")
print()

# Init Ray
if ray.is_initialized():
    ray.shutdown()
ray.init(ignore_reinit_error=True)

# Build algorithm
env_config = {
    "track_paths": track_paths,
    "min_horse_count": MIN_HORSES,
    "max_horse_count": MAX_HORSES,
    "max_steps": MAX_STEPS,
    "randomize_archetypes": True,
    "randomize_genomes": True,
}

config = (
    PPOConfig()
    .environment(
        env=HorseRacingRLlibEnv,
        env_config=env_config,
    )
    .env_runners(
        num_env_runners=num_workers,
        num_envs_per_env_runner=1,
    )
    .training(
        train_batch_size=16000,
        minibatch_size=512,
        num_epochs=10,
        lr=3e-4,
        gamma=0.995,
        lambda_=0.95,
        clip_param=0.2,
        vf_clip_param=50.0,
        entropy_coeff=0.005,
    )
    .rl_module(
        model_config={
            "fcnet_hiddens": [256, 256],
            "fcnet_activation": "relu",
        },
    )
    .resources(
        num_gpus=num_gpus,
    )
    .framework("torch")
    .multi_agent(
        policies={"shared_policy": PolicySpec()},
        policy_mapping_fn=lambda agent_id, *args, **kwargs: "shared_policy",
    )
)

algo = config.build_algo()

# Restore from checkpoint if available
if checkpoint_path:
    algo.restore(checkpoint_path)
    print(f"Restored from: {checkpoint_path}")

print("Algorithm ready.\n")

In [ ]:
# --- Training loop ---
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

best_reward = float("-inf")
total_timesteps = 0
iterations = 0
start_time = time.time()
history = []

print(f"Training to {TOTAL_TIMESTEPS:,} timesteps...")
print(f"Checkpoints → {CHECKPOINT_DIR}\n")

try:
    while total_timesteps < TOTAL_TIMESTEPS:
        result = algo.train()
        iterations += 1

        # Extract metrics (Ray 2.40+)
        er = result.get("env_runners", {})
        mean_reward = er.get("episode_return_mean", er.get("episode_reward_mean", 0.0))
        max_reward = er.get("episode_return_max", er.get("episode_reward_max", 0.0))
        ep_len = er.get("episode_len_mean", 0.0)
        num_episodes = er.get("num_episodes", 0)
        total_timesteps = result.get(
            "num_env_steps_sampled_lifetime",
            er.get("num_env_steps_sampled_lifetime", 0),
        )

        # Per-agent rewards (for archetype diversity tracking)
        agent_rewards = er.get("agent_episode_returns_mean", {})

        history.append({
            "iter": iterations,
            "timesteps": total_timesteps,
            "mean_reward": float(mean_reward),
            "max_reward": float(max_reward),
            "ep_len": float(ep_len),
            "episodes": int(num_episodes),
        })

        if iterations % LOG_EVERY == 0 or iterations == 1:
            elapsed = time.time() - start_time
            rate = total_timesteps / elapsed if elapsed > 0 else 0
            pct = total_timesteps / TOTAL_TIMESTEPS * 100
            print(
                f"  [{pct:5.1f}%] Iter {iterations:4d} | "
                f"ts={total_timesteps:>10,} | "
                f"reward={mean_reward:8.1f} (max={max_reward:8.1f}) | "
                f"ep_len={ep_len:5.0f} | "
                f"{rate:,.0f} ts/s"
            )

        # Save best
        if mean_reward > best_reward:
            best_reward = mean_reward
            algo.save(os.path.join(CHECKPOINT_DIR, "best"))
            if iterations % LOG_EVERY == 0:
                print(f"  *** New best: {best_reward:.1f}")

        # Periodic checkpoint (to Drive)
        if iterations % SAVE_EVERY == 0:
            save_path = algo.save(os.path.join(CHECKPOINT_DIR, f"iter_{iterations}"))
            print(f"  Checkpoint saved: iter_{iterations}")

            # Also save history
            with open(os.path.join(CHECKPOINT_DIR, "history.json"), "w") as f:
                json.dump(history, f)

except KeyboardInterrupt:
    print("\nTraining interrupted.")

# Final save
final_path = algo.save(os.path.join(CHECKPOINT_DIR, "latest"))
with open(os.path.join(CHECKPOINT_DIR, "history.json"), "w") as f:
    json.dump(history, f)

elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"Training complete")
print(f"  Timesteps: {total_timesteps:,}")
print(f"  Iterations: {iterations}")
print(f"  Best reward: {best_reward:.1f}")
print(f"  Time: {elapsed/60:.1f} min")
print(f"  Checkpoint: {CHECKPOINT_DIR}/latest")
print(f"{'='*60}")

## Export to ONNX

In [ ]:
import torch.nn as nn
import numpy as np
import onnx
import onnxruntime as ort
from horse_racing.types import OBS_SIZE

class PolicyWrapper(nn.Module):
    """Wraps RLlib RLModule for ONNX export: obs -> action means."""
    def __init__(self, rl_module):
        super().__init__()
        self.rl_module = rl_module
    def forward(self, obs):
        result = self.rl_module.forward_inference({"obs": obs})
        return result["action_dist_inputs"][:, :2]  # action means only

# Extract and wrap the shared policy
module = algo.get_module("shared_policy")
module.eval()
wrapper = PolicyWrapper(module)
wrapper.eval()

# Export
onnx_path = os.path.join(CHECKPOINT_DIR, "horse_jockey.onnx")
dummy = torch.zeros(1, OBS_SIZE, dtype=torch.float32)

with torch.no_grad():
    test_out = wrapper(dummy)
    print(f"Test output: {test_out.numpy()}")

torch.onnx.export(
    wrapper, dummy, onnx_path,
    input_names=["obs"],
    output_names=["action"],
    dynamic_axes={"obs": {0: "batch"}, "action": {0: "batch"}},
    opset_version=17,
    dynamo=False,
)

# Verify
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
session = ort.InferenceSession(onnx_path)
result = session.run(None, {"obs": np.zeros((1, OBS_SIZE), dtype=np.float32)})
print(f"ONNX output: {result[0]}")
print(f"Exported to: {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)")

## Quick Eval Race

In [ ]:
# Run a quick 6-horse race using the trained model
from horse_racing.engine import EngineConfig, HorseRacingEngine
from horse_racing.types import HorseAction

session = ort.InferenceSession(onnx_path)
engine = HorseRacingEngine("tracks/tokyo.json", EngineConfig(horse_count=6))

for step in range(5000):
    obs_list = engine.get_observations()
    obs_arrays = np.array([engine.obs_to_array(o) for o in obs_list], dtype=np.float32)
    actions = session.run(None, {"obs": obs_arrays})[0]

    action_list = [HorseAction(float(a[0]), float(a[1])) for a in actions]
    engine.step(action_list)

    if all(h.finished for h in engine.horses):
        break

# Results
placements = engine.get_placements()
print(f"Race finished in {step+1} steps\n")
print(f"{'Horse':>8} {'Place':>6} {'Progress':>10} {'Stamina':>10} {'Finished':>10}")
print("-" * 50)
for i, hs in enumerate(engine.horses):
    stam = hs.runtime.current_stamina / hs.base_attrs.stamina if hs.base_attrs.stamina > 0 else 0
    print(f"  H{i:>5} {placements[i]:>6} {hs.track_progress:>10.3f} {stam:>10.2f} {'Yes' if hs.finished else 'No':>10}")

algo.stop()
ray.shutdown()
print("\nDone.")